#Cell 1: Environment Setup and Initialization
In this cell, we install (if necessary) and import the core libraries. We use set_seed to ensure that our random text generations are reproducible. We also detect if a GPU (CUDA) is available to speed up the generation process.

In [5]:
from transformers import pipeline, set_seed, GPT2LMHeadModel, GPT2Tokenizer
import torch

# Reproducibility for exact generations
set_seed(42)

# Check for GPU
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

Using device: GPU


#Cell 2: Text Generation via the Pipeline API
The pipeline is the easiest way to use a pre-trained model. Here, we load the gpt2 model for the task of text-generation. This API handles tokenization, model inference, and decoding automatically. We set parameters like temperature (to control randomness) and top_p (to ensure the model chooses from the most likely words).

In [6]:
from transformers import pipeline

# Initialize the high-level pipeline
generator = pipeline('text-generation', model='gpt2', device=device)

prompt = "Once upon a time in a distant galaxy,"

# Generate 3 variations of the story
outputs = generator(prompt, max_new_tokens=100, num_return_sequences=3,
                    temperature=0.8, top_p=0.95, do_sample=True, pad_token_id=50256)

print("=== Text Generations from Pipeline ===")
for i, output in enumerate(outputs):
    print(f"\nGeneration {i+1}:\n{output['generated_text']}")
    print("-" * 80)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'pad_token_id', 'num_return_sequences', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Text Generations from Pipeline ===

Generation 1:
Once upon a time in a distant galaxy, a vast galaxy that had been destroyed, and had been completely destroyed, the world of the human race seemed to be on the brink of destruction. The human race had not yet arrived at its final destination.

Now, as it slowly turned its attention to the next step, the human race had discovered something that had surprised everyone.

A small, dark, black hole.

A world in which the humans and the aliens had been separated.

A world in which the humans
--------------------------------------------------------------------------------

Generation 2:
Once upon a time in a distant galaxy, the galactic community had already developed a technological civilization capable of living in harmony with the cosmos, but now, many years after it had gained control over its own species, the species' species had been wiped out. The species' collective consciousness had been restored, but it was still not entirely fun

#Cell 3: Loading the Tokenizer and Model Manually
For more complex tasks, you might want to interact with the Tokenizer and Model separately. The Tokenizer converts text into numerical IDs (tokens) that the model understands, while the Model processes those numbers. We explicitly set the pad_token to the end-of-sentence token to avoid errors during generation.

In [3]:
# Manual approach: Loading specific components
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2').to('cuda' if device == 0 else 'cpu')

# Ensure the tokenizer has a padding token defined
tokenizer.pad_token = tokenizer.eos_token

print("Model and Tokenizer loaded manually.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model and Tokenizer loaded manually.


#Cell 4: Manual Inference and Decoding
In this final cell, we manually convert a text prompt into tensors (numerical arrays) and pass them to the model.generate() method. This gives us access to advanced parameters like repetition_penalty, which prevents the model from repeating the same phrases. Finally, we "decode" the numerical output back into human-readable text.

In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Check for GPU (from original Cell 1)
device = 0 if torch.cuda.is_available() else -1

# Manual approach: Loading specific components (from original Cell 3)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2').to('cuda' if device == 0 else 'cpu')

# Ensure the tokenizer has a padding token defined
tokenizer.pad_token = tokenizer.eos_token

prompt2 = "In a world where AI has become conscious,"

# Tokenize input text
inputs = tokenizer(prompt2, return_tensors="pt").to(model.device)

# Generate output with fine-tuned parameters
outputs2 = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.2
)

print("\n=== Manual Generation with Full Control ===")
print(tokenizer.decode(outputs2[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



=== Manual Generation with Full Control ===
In a world where AI has become conscious, we should be concerned about our ability to predict and respond appropriately.
The new research was done by the University of Cambridge's Computer Science Institute (CSI) in collaboration with Professor John Lasseter from CSIS' Faculty Development Centre for Human Factors Research at CSE who led this work on Wednesday 16 December 2016 using data collected during an online survey conducted between November 2015-December 2017 as part Ofc1aE3A/2AIF4B6C9: http://www0xa09zgjrxqpwdwpvf8o7b_t
